<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>



# Lane Control

Now that we have an estimate of our lateral displacement $d$ and our angle relative to the lane $\phi$, our task here is to decide how to control the wheels so that the robot drives straight down the lane. Even if we place the robot directly in the middle of the lane, simply commanding it to go straight (equal forward speed on each wheel) will obviously not work because: 1) there are disturbances and the robot will deviate (we mitigate this as much as possible by the process of [wheel calibration](https://docs.duckietown.com/ente/duckietown-manual/20-operations/04-calibrations/duckiebot-motor-odometry-calibration.html) but it will never be perfect) and 2) there are turns in the road that we need to adapt to. 

We will need to use our information about $d$ and $\phi$ to set the speeds of the wheels, which is feedback control. 

## Inverse Kinematics

One first note is that it will be intuitively simpler to build our controller over the linear speed ($v$) and the angular speed ($\omega$) of the Duckiebot since we can easily convert these to right and left wheel speeds using the _inverse kinematics_ equations. For more details see the [kinematics and odometry LX](https://github.com/duckietown/lx-kinematics-odometry). 

## "PID-Inspired" Control

In the [Duckietown control LX](https://docs.duckietown.com/ente/duckietown-manual/60-learning-experiences/lx-setup-pid-control.html) we have learned about a very simple feedback control strategy called "PID", where the control is a linear combination of the proportial error, the change in error (derivative) and the accumulated error (integral). The was simple to implement when our state was one dimensional, and our control was one-dimensional. In this case we have two states ($d$ and $phi$) and two control outputs ($v$ and $\omega$). We can reduce the control outputs to one dimension by simply assuming that we want the robot to travel at a fixed velocity $v=\bar{v}$ and only worry about how to control the steering of the robot through $\omega$. 

We subsequently propose the following control law:

$$
u_t = K^d_d d_t + K^d_{\phi} \phi_t + K^i_d d_t,
$$

where:

* $K^d_d$ is the proportional gain weighting the diplacement error.
* $K^d_\phi$ is the proportional gain weighting the angular error.
* $K^I_d$ is the integral gain weighting the accumulated diplacement error.

We should note that, although this closely resembles a PID controller, it technically is not one. In fact this controller can be derived from a "pole-placement" style of approach which is beyong the scope of this notebook but for details please refer to [these slides](https://docs.google.com/presentation/d/15awvCABgWQhjMkF6bMfAIevz-468u92V/edit?slide=id.p29#slide=id.p29).



## Tuning the Gains

Tuning the controller gains is one of the most important aspects for the stability and performance of your Duckiebot.

We can largely follow the strategy introduced in the [control learning experience](https://github.com/duckietown/lx-control), if we consider that our reference is the middle of the road ($d=0$) and that, for relatively small values of $\phi$, the $\phi$ error acts similarly to a derivative term for tracking $d=0$. Put another way, we can intuitively understand that if the robot is not directly pointing down the lane ($\phi \neq 0$) then this will induce a change in the lateral displacement error, which is very similar to what the derivative term does in a PID controller. So, following the PID intuition we should expect that: 

* The $K^d_d$ gain affects the magnitude of corrections. Too high a value leads to oscillations, while too low a value results in sluggish response (similar to the proportonal gain in a PID controller);
* The $K^d_\phi$ gain smooths out the response by reducing overshoot but can amplify noise (similar to the derivative gain in a PID controller);
* The $K^I_d$ gain reduces steady-state errors but can introduce instability if over-tuned (similar to the integral gain in a PID controller).

These are located in the config for the [lane controller node](../packages/dt-core/packages/lane_control/config/lane_controller_node/). Here are the defaults:


```yaml 
v_bar: 0.19 #constant forward velocity (if gain k=1 in odometry calibration) 
k_d_d: -2.0 # as above 
k_d_phi: -3.0 # as above 
k_I_d: -3.0 # as above

integral_bounds: 
  d:
    top: 0.3
    bot: -0.3
```

The are defined as follows: 

- `v_bar`:  Nominal forward velocity (m/s). Note that this is also dependant on the `gain` that is tuned during [wheel calibration](https://docs.duckietown.com/ente/duckietown-manual/20-operations/04-calibrations/duckiebot-motor-odometry-calibration.html). 


- `k_d_d`: Proportional gain on lateral error $d$. Negative because a positive lateral deviation (too far left) should produce a negative angular velocity (turn right). Larger magnitude = more aggressive lateral correction.

- `k_d_phi`:  Proportional gain on heading error `$\phi$`. Negative for the same sign convention reason. Typically larger than `k_d_d` since heading error is easier to correct quickly.


- `k_I_d`:  Integral gain on lateral error. Accumulates d error over time to eliminate steady-state lateral offset (e.g. a consistent bias calibration error).

- `integral_bounds`: Anti-windup limits for the lateral integral term. The accumulated `$d$` error is clamped to this range to prevent integrator windup when the robot is held off-center for a long time.



For more details about PID control please refer to the [Control Learning Experience](https://docs.duckietown.com/ente/duckietown-manual/60-learning-experiences/lx-setup-pid-control.html).

The velocity and steering values are turned into actuator values using inverse kinematics by [the kinematics node](../packages/dt-core/packages/robots/duckiebot/dagu_car/src/kinematics_node.py). For more details about direct and inverse kinematics refer to the [Kinematics and Odometry Learning Experience](https://docs.duckietown.com/ente/duckietown-manual/60-learning-experiences/lx-setup-modeling-kinematics.html).

We now have understood the entire autonomy stack, from the data that comes in through the sensors (camera and encoders) to the actuator commands that are sent to the wheels. One last piece to discuss is how we build a "finite state machine" that sits on top of this stack and manages what the macro-level behaviour of the robot should be. 

In this relatively simple autonomous behavior case we have two (autonomy) states: `LANE_FOLLOWING`, and `NORMAL_JOYSTICK_CONTROL` that can  be toggled using the `keyboard_control` GUI. For details, proceed to the [the next notebook about the finite state machine](./05_finite_state_machine.ipynb).